# Data Wrangling

In this notebook, we will load raw data: Stanford Open Police Project and Social Vulnerability Index, and conduct merging.

## Set up

In [32]:
# Imports
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import pyogrio
import sys
import subprocess
import warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # if running from notebooks/
from paths import *

In [33]:
# Install requirement.txt
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

CompletedProcess(args=['D:\\PythonCode\\python.exe', '-m', 'pip', 'install', '-r', '../requirements.txt'], returncode=0)

## Read data

Police stop and Social Vulnerability datasets

In [34]:
sopp = pd.read_csv(RAW_DATA_PATH / "sopp.csv") # police stop dataset
svi_tab = pd.read_csv(RAW_DATA_PATH / "California.csv")# social vulnerability index dataset

svi_tab = svi_tab[['FIPS', 'RPL_THEMES']].copy() # These are the columns for merging later
svi_tab['FIPS'] = svi_tab['FIPS'].astype(str).str.replace(r'\.0$', '', regex=True) # geographical area identifier

# Preview datasets
print(sopp.head())
print(svi_tab.head())

  raw_row_number        date      time service_area  subject_age  \
0              1  2014-01-01  01:25:00          110         24.0   
1              2  2014-01-01  05:47:00          320         42.0   
2              3  2014-01-01  07:46:00          320         29.0   
3              4  2014-01-01  08:10:00          610         23.0   
4              5  2014-01-01  08:35:00          930         35.0   

             subject_race subject_sex       type arrest_made citation_issued  \
0                   white        male  vehicular       False            True   
1                   white        male  vehicular       False           False   
2  asian/pacific islander        male  vehicular       False           False   
3                   white        male  vehicular       False            True   
4                hispanic        male  vehicular       False            True   

   ...   outcome contraband_found search_conducted  search_person  \
0  ...  citation              NaN        

Shape files for police stop and Social Vulnerability datasets

In [35]:
# Some info about the police service zones
beats = gpd.read_file(RAW_DATA_PATH / "Police_Beats.geojson")
print(beats.head()) # preview police service zones
print(beats.columns.tolist()) # get columns of police service zones
print(beats.crs) # crs is a framework used to define how 3D spatial data (locations on the Earth's surface) 

   objectid  beat  div  serv        name  \
0         9   935    9   930  NORTH CITY   
1        13     0    0     0   SAN DIEGO   
2        14   511    5   510        None   
3        15   722    7   720      NESTOR   
4        16   314    3   310    BIRDLAND   

                                            geometry  
0  POLYGON ((-117.23876 32.98575, -117.2387 32.98...  
1  MULTIPOLYGON (((-117.22526 32.70267, -117.2252...  
2  MULTIPOLYGON (((-117.22529 32.7026, -117.22525...  
3  POLYGON ((-117.09042 32.58382, -117.09001 32.5...  
4  POLYGON ((-117.15149 32.8065, -117.1514 32.806...  
['objectid', 'beat', 'div', 'serv', 'name', 'geometry']
EPSG:4326


In [36]:
# Suppress pyogrio/GeoPandas user warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pyogrio")

# Filter for California state in census tract
layer_name = "CALIFORNIA_tract"
tracts = gpd.read_file(RAW_DATA_PATH / "California/SVI2014_CALIFORNIA_tract.gdb", layer=layer_name)

# Preview SVI's census tract geometry
print(tracts.head()) # see the first several rows of census tract geometry
print(tracts.columns.tolist()) # get columns of census tract geometry
print(tracts.crs) # crs is a framework used to define how 3D spatial data (locations on the Earth's surface) 

               AFFGEOID TRACTCE  ST        STATE ST_ABBR STCNTY    COUNTY  \
0  1400000US06001400100  400100  06   California      CA  06001   Alameda   
1  1400000US06001400200  400200  06   California      CA  06001   Alameda   
2  1400000US06001400300  400300  06   California      CA  06001   Alameda   
3  1400000US06001400400  400400  06   California      CA  06001   Alameda   
4  1400000US06001400500  400500  06   California      CA  06001   Alameda   

          FIPS                                       LOCATION  AREA_SQMI  ...  \
0  06001400100  Census Tract 4001, Alameda County, California   2.661917  ...   
1  06001400200  Census Tract 4002, Alameda County, California   0.226817  ...   
2  06001400300  Census Tract 4003, Alameda County, California   0.426770  ...   
3  06001400400  Census Tract 4004, Alameda County, California   0.275958  ...   
4  06001400500  Census Tract 4005, Alameda County, California   0.227919  ...   

   F_THEME4  F_TOTAL  E_UNINSUR  M_UNINSUR  EP_UNI

In [37]:
# Standardize FIPS code
tracts['FIPS'] = tracts['FIPS'].astype(str).str.replace(r'\.0$', '', regex=True)
tracts_sd = tracts[tracts['FIPS'].str.startswith('06073')].copy() # get a copy of tracts in San Diego area
tracts_sd = tracts_sd[['FIPS', 'RPL_THEMES', 'geometry']].copy() # filter columns to contain FIPS, SVI index, and geometric details

print(tracts_sd[['FIPS', 'RPL_THEMES']].head()) # preview FIPS and SVI index in San Diego
print("Rows in San Diego tracts:", len(tracts_sd)) # outputs number of tracts in San Diego

             FIPS  RPL_THEMES
4931  06073008509      0.4912
4932  06073008510      0.4610
4933  06073008511      0.4802
4934  06073008512      0.1664
4935  06073008513      0.0467
Rows in San Diego tracts: 627


## Merge datasets

Now match the police zones and census tracts. 

The set $A$ is the set of policing zones (it is made up of some smaller sets, $A_j$, corresponding to each zone in particular).

The set $B$ is the set of all census tracts (also made up of sets $B_j$). 

$A$ and $B$ are both subsets of an area (for example, San Diego). We want to match the data where $A \cap B$ where the police zones intersect with the census tracts on social vulnerability.

Ensure both datasets use the SAME coordinate system

In [38]:
# Ensure both datasets use the SAME coordinate system (CRS)
if beats.crs != tracts_sd.crs:
    tracts_sd = tracts_sd.to_crs(beats.crs)

# Convert both datasets to a projected CRS (units = meters)
beats_proj = beats.to_crs(epsg=26911)      # UTM Zone 11N (good for California)
tracts_proj = tracts_sd.to_crs(epsg=26911)

# Perform spatial intersection
inter = gpd.overlay(beats_proj, tracts_proj, how="intersection")

# Compute the area of each overlapping piece (in square meters)
inter["overlap_area"] = inter.geometry.area 

# Compute weighted SVI for each overlap piece
# RPL_THEMES = SVI score → multiply by area to weight it
inter["weighted_svi"] = inter["RPL_THEMES"] * inter["overlap_area"]

# Preview the resulting dataset
print(inter.head())

   objectid  beat  div  serv        name         FIPS  RPL_THEMES  \
0         9   935    9   930  NORTH CITY  06073017029      0.0102   
1         9   935    9   930  NORTH CITY  06073017106      0.1011   
2         9   935    9   930  NORTH CITY  06073017304      0.3095   
3         9   935    9   930  NORTH CITY  06073017306      0.0422   
4         9   935    9   930  NORTH CITY  06073008324      0.0189   

                                            geometry  overlap_area  \
0  MULTIPOLYGON Z (((479103.087 3649461.789 0, 47...  2.490648e+06   
1  MULTIPOLYGON Z (((478835.266 3649610.423 0, 47...  1.821234e+04   
2  POLYGON Z ((476337.773 3649093.502 0, 476337.4...  5.891610e+04   
3  MULTIPOLYGON Z (((477697.717 3649731.986 0, 47...  6.566568e+05   
4  POLYGON Z ((476212.978 3649065.552 0, 476210.0...  2.955893e+02   

   weighted_svi  
0  25404.609253  
1   1841.267895  
2  18234.534197  
3  27710.918559  
4      5.586638  


Group the intersection data by service area

In [40]:
# Group the intersection data by service area
# For each group, compute total weighted SVI (sum of svi * area) and Total overlap area
svc_svi = (
    inter.groupby("serv", dropna=False)
    .agg(
        weighted_svi_sum=("weighted_svi", "sum"),   # sum of (SVI * area)
        overlap_area_sum=("overlap_area", "sum")    # total area in this zone
    )
    .reset_index()  # turn groupby result back into a normal DataFrame
)

# Compute final area-weighted SVI for each service area
svc_svi["svi_rpl_themes"] = svc_svi["weighted_svi_sum"] / svc_svi["overlap_area_sum"]

# Keep only relevant columns: service area ID + final SVI score
svc_svi = svc_svi[["serv", "svi_rpl_themes"]]

# Preview results
print(svc_svi.head())

   serv  svi_rpl_themes
0     0        0.029338
1   110        0.241848
2   120        0.143232
3   230        0.162551
4   240        0.131157


Make service_area column type consistent and merge

In [25]:
# Convert service_area to string so it matches the type in svc_svi
sopp["service_area"] = sopp["service_area"].astype(str)

# Convert "serv" column (from SVI results) to string
svc_svi["serv"] = svc_svi["serv"].astype(str)

# Merge service_area (in sopp) with serv (in svc_svi) using left join
sopp_final = sopp.merge(
    svc_svi,
    left_on="service_area",
    right_on="serv",
    how="left"
)

# Preview key columns:
print(sopp_final[["date", "service_area", "search_conducted", "svi_rpl_themes"]].head())

# Count missing matches
print("Missing SVI rows:", sopp_final["svi_rpl_themes"].isna().sum())

         date service_area  search_conducted  svi_rpl_themes
0  2014-01-01          110             False        0.241848
1  2014-01-01          320             False        0.213643
2  2014-01-01          320             False        0.213643
3  2014-01-01          610             False        0.121181
4  2014-01-01          930             False        0.075382
Missing SVI rows: 11627


Export the intermediate data set.

In [26]:
sopp_final.to_csv(INT_DATA_PATH / "sopp_svi_merged.csv")